In [6]:
!ls "/content/drive/MyDrive/TrashNeXt Dataset.zip"


'/content/drive/MyDrive/TrashNeXt Dataset.zip'


In [1]:
!pip install timm transformers datasets accelerate --quiet

In [5]:
import zipfile
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
zip_path="/content/drive/MyDrive/TrashNeXt Dataset.zip"
extract_path = "/content"



In [8]:
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)
!ls {extract_path}

Dataset extracted to: /content
 drive	 __MACOSX   sample_data  'TrashNeXt Dataset'


In [ ]:
import os
from PIL import Image
from tqdm import tqdm

def is_corrupted_image(file_path):
    """Check if an image file is corrupted."""
    try:
        with Image.open(file_path) as img:
            img.verify()  # Verify integrity
        return False
    except (IOError, SyntaxError, Image.DecompressionBombError):
        return True

def remove_corrupted_images(dataset_path):
    """Scan and remove corrupted images + hidden macOS files."""
    corrupted_count = 0
    total_images = 0

    for root, _, files in os.walk(dataset_path):
        for file in tqdm(files, desc=f"Scanning {os.path.basename(root)}"):

            # Skip hidden files (.DS_Store, ._files)
            if file.startswith("."):
                file_path = os.path.join(root, file)
                try:
                    os.remove(file_path)
                    corrupted_count += 1
                    print(f"Removed hidden file: {file_path}")
                except:
                    pass
                continue

            # Check only image formats
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                total_images += 1
                file_path = os.path.join(root, file)
                if is_corrupted_image(file_path):
                    try:
                        os.remove(file_path)
                        corrupted_count += 1
                        print(f"Removed corrupted image: {file_path}")
                    except Exception as e:
                        print(f"Error removing {file_path}: {e}")

    print("\n✅ Scan completed!")
    print(f"📷 Total images scanned: {total_images}")
    print(f"🗑️ Corrupted/hidden files removed: {corrupted_count}")

# Run the cleaner
dataset_path = "/content/TrashNeXt Dataset"
remove_corrupted_images(dataset_path)


In [ ]:
from PIL import Image
import os

def remove_truncated_images(dataset_path):
    removed_count = 0  # Counter for removed images

    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(root, file)
                try:
                    img = Image.open(img_path)
                    img.verify()  # Verify if the image is not corrupted
                    img.close()   # Close the image to avoid resource leaks
                except Exception as e:
                    print(f"Removing corrupted/truncated: {img_path}")
                    os.remove(img_path)
                    removed_count += 1

    print(f"\nTotal corrupted/truncated images removed: {removed_count}")

# Example usage
dataset_path = "/content/TrashNeXt Dataset"
remove_truncated_images(dataset_path)

In [ ]:
import os

dataset_path = "/content/TrashNeXt Dataset"
splits = ["Train", "Valid", "Test"]

for split in splits:
    print(f"\n{split.upper()} Split:")
    split_path = os.path.join(dataset_path, split)

    for cls in sorted(os.listdir(split_path)):
        cls_path = os.path.join(split_path, cls)
        if os.path.isdir(cls_path):
            num_images = len([f for f in os.listdir(cls_path) if not f.startswith(".")])
            print(f"  {cls}: {num_images} images")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# 🔹 Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 🔹 Paths
train_dir = "/content/TrashNeXt Dataset/Train"
valid_dir = "/content/TrashNeXt Dataset/Valid"
test_dir  = "/content/TrashNeXt Dataset/Test"

# 🔹 Use official preprocessing from pretrained weights
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

# 🔹 Datasets
train_dataset = datasets.ImageFolder(train_dir, transform=preprocess)
valid_dataset = datasets.ImageFolder(valid_dir, transform=preprocess)
test_dataset  = datasets.ImageFolder(test_dir,  transform=preprocess)

# 🔹 Optimized DataLoaders
num_workers = os.cpu_count()
common = dict(batch_size=32, num_workers=num_workers, pin_memory=True, persistent_workers=True)

train_loader = DataLoader(train_dataset, shuffle=True, **common)
valid_loader = DataLoader(valid_dataset, shuffle=False, **common)
test_loader  = DataLoader(test_dataset,  shuffle=False, **common)

# Print class names
print(f"Classes: {train_dataset.classes}")
print(f"Number of classes: {len(train_dataset.classes)}")


In [ ]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained EfficientNetV2-S
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = efficientnet_v2_s(weights=weights)

# Replace classifier for your dataset
num_classes = len(train_dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Send to device
model = model.to(device)

print(f"✅ Model initialized for {num_classes} classes on {device}")


In [ ]:
# 🔹 Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
# 🔹 Training Loop
epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_acc = 100. * correct / total

    # 🔹 Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100. * val_correct / val_total
    print(f"\nEpoch [{epoch+1}/{epochs}] Train Loss: {running_loss/len(train_loader):.4f} "
          f"Train Acc: {train_acc:.2f}% Valid Acc: {val_acc:.2f}%")

In [ ]:
# 🔹 Test Evaluation (EfficientNetV2)
model.eval()
test_correct, test_total = 0, 0
test_loss = 0.0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, preds = outputs.max(1)
        test_correct += preds.eq(labels).sum().item()
        test_total += labels.size(0)

test_acc = 100. * test_correct / test_total
print(f"\nTest Loss: {test_loss/len(test_loader):.4f} | Test Accuracy: {test_acc:.2f}%")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 🔹 Collect predictions & labels
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 🔹 Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.classes)

# 🔹 Plot
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap="Blues", ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix - EfficientNetV2")
plt.show()


In [ ]:
# 🔹 Switch to eval mode
model.eval()

# 🔹 Example input for tracing (match input size: 300x300 for EfficientNetV2)
example_input = torch.randn(1, 3, 300, 300).to(device)

# 🔹 Trace the model (strict=False allows more flexibility)
traced_model = torch.jit.trace(model, example_input, strict=False)

# 🔹 Save TorchScript
traced_model.save("efficientnetv2_trashnext_rpi.pt")

print("✅ TorchScript model saved as efficientnetv2_trashnext_rpi.pt")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 🔹 Collect predictions and true labels
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 🔹 Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# 🔹 Class-wise accuracy
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\nClass-wise Accuracy:")
for cls, acc in zip(train_dataset.classes, class_accuracy):
    print(f"{cls}: {acc*100:.2f}%")

# 🔹 Full classification report (precision, recall, f1-score)
print("\nDetailed Report:")
print(classification_report(all_labels, all_preds, target_names=train_dataset.classes))
